# Invoice Extractor — Azure Document Intelligence + Azure OpenAI

**Pipeline:**
1. **prebuilt-layout** OCR → extract clean text per page (handles multi-invoice PDFs correctly)
2. **OpenAI per-page** → identify & extract every invoice on each page (handles multi-page invoices + multiple invoices per page)
3. **Deduplicate** → group multi-page invoices by invoice number, record all page numbers
4. **Enrich** → OpenAI generates `invcat` + `invdec` in parallel batches
5. **Save** → CSV with columns: `sr`, `invdt`, `invto`, `invnum`, `invamt`, `pgnum`, `invcat`, `invdec`


In [1]:
# ── Cell 1 · Install dependencies ────────────────────────────────────────────
%pip install azure-ai-documentintelligence azure-identity openai pandas tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# ── Cell 2 · Configuration ────────────────────────────────────────────────────
import os

# ── Azure Document Intelligence ──
DOC_INTEL_ENDPOINT = os.environ.get("DOC_INTEL_ENDPOINT", "")
DOC_INTEL_KEY      = os.environ.get("DOC_INTEL_KEY", "")

# ── Azure OpenAI ──
AOAI_ENDPOINT    = os.environ.get("AOAI_ENDPOINT", "")
AOAI_KEY         = os.environ.get("AOAI_KEY", "")
AOAI_DEPLOYMENT  = "gpt-4.1-mini"
AOAI_API_VERSION = "2024-02-01"

# ── PDF file path ──
PDF_PATH = r"C:\Users\KR614XU\Downloads\Ishaan\Test Files\GC GR Feb 2026.pdf"

# ── Mode ──
TEST_MODE  = False
TEST_PAGES = 20

# ── Output ──
_suffix    = f"_test{TEST_PAGES}pages" if TEST_MODE else "_full"
OUTPUT_CSV = rf"C:\Users\KR614XU\Downloads\Ishaan\invoices_output{_suffix}.csv"

# ── Extraction tuning ──
PAGE_BATCH_SIZE  = 5
PAGE_TEXT_LIMIT  = 5000   # ← raised from 2500: captures full invoice content incl. totals on long pages
EXTRACT_WORKERS  = 10
BATCH_INTER_WAIT = 0

# ── Enrichment tuning ──
BATCH_SIZE = 20

pdf_size_mb = os.path.getsize(PDF_PATH) / 1_048_576
print(f"Model         : {AOAI_DEPLOYMENT}  (non-reasoning — fast & accurate)")
print(f"TPM limit     : 200,000")
print(f"PDF           : {os.path.basename(PDF_PATH)}  ({pdf_size_mb:.1f} MB)")
print(f"Mode          : {'TEST — first ' + str(TEST_PAGES) + ' pages' if TEST_MODE else 'FULL PDF (all 808 pages)'}")
print(f"Output CSV    : {OUTPUT_CSV}")
print(f"Extraction    : {PAGE_BATCH_SIZE} pages/batch · {PAGE_TEXT_LIMIT} chars/page · {EXTRACT_WORKERS} workers")
print()
total_batches_est = (808 if not TEST_MODE else TEST_PAGES) // PAGE_BATCH_SIZE
print(f"Estimated batches : {total_batches_est}")
print(f"Estimated time    : ~2-5 min")


Model         : gpt-4.1-mini  (non-reasoning — fast & accurate)
TPM limit     : 200,000
PDF           : GC GR Feb 2026.pdf  (60.5 MB)
Mode          : FULL PDF (all 808 pages)
Output CSV    : C:\Users\KR614XU\Downloads\Ishaan\invoices_output_full.csv
Extraction    : 5 pages/batch · 5000 chars/page · 10 workers

Estimated batches : 161
Estimated time    : ~2-5 min


In [4]:
# ── Cell 3 · Step 1 — Extract page-by-page text via prebuilt-layout ───────────
# WHY layout instead of invoice: prebuilt-invoice treats a whole multi-invoice
# PDF as one document. prebuilt-layout gives us clean text PER PAGE which we
# then feed to OpenAI to find every individual invoice.

from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential
import json

pages_param = f"1-{TEST_PAGES}" if TEST_MODE else None

print("=" * 60)
print(f"  Model    : prebuilt-layout  (page-by-page OCR)")
print(f"  File     : {PDF_PATH}")
print(f"  Pages    : {pages_param if pages_param else 'ALL'}")
print("=" * 60)

di_client = DocumentIntelligenceClient(
    endpoint=DOC_INTEL_ENDPOINT,
    credential=AzureKeyCredential(DOC_INTEL_KEY)
)

with open(PDF_PATH, "rb") as f:
    pdf_bytes = f.read()

poller = di_client.begin_analyze_document(
    model_id="prebuilt-layout",
    body=AnalyzeDocumentRequest(bytes_source=pdf_bytes),
    pages=pages_param
)

print("Waiting for Document Intelligence to finish…")
result = poller.result()

# ── Build page_texts: {page_number: "full text of that page"} ──
page_texts = {}
if result.pages:
    for pg in result.pages:
        lines = [line.content for line in (pg.lines or [])]
        page_texts[pg.page_number] = "\n".join(lines)

print(f"\nDone! Pages extracted: {len(page_texts)}")
for pnum, txt in list(page_texts.items())[:3]:
    preview = txt[:120].replace("\n", " ")
    print(f"  Page {pnum}: {preview}…")


  Model    : prebuilt-layout  (page-by-page OCR)
  File     : C:\Users\KR614XU\Downloads\Ishaan\Test Files\GC GR Feb 2026.pdf
  Pages    : ALL
Waiting for Document Intelligence to finish…

Done! Pages extracted: 808
  Page 1: ez cater ezCater, Inc 40 Water Street 5th Floor Boston, MA 02109 Bill To : HITT Contracting, Inc. 2900 Fairview Park Dr.…
  Page 2: Order Details Order Number Placed By Address Order Amount Order Tax GPJ6MU Administrative Services 2900 Fairview Park Dr…
  Page 3: Duncan Parnell PO Box 604176 Charlotte, NC 28260-4176 Phone: 704-526-1883 arsupport@Duncan-Parnell.com Branch: 04-North …


In [20]:
# ── Cell 4 · Step 2 — Batch per-page OpenAI extraction ───────────────────────
import concurrent.futures
import time
import json
import os
from openai import AzureOpenAI
from tqdm import tqdm
from collections import defaultdict

aoai_client = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_KEY,
    api_version=AOAI_API_VERSION
)

PAGE_EXTRACT_SYSTEM = """You are an expert expense data extractor for a construction company.
You will receive OCR text from multiple consecutive PDF pages, each labelled === PAGE N ===.
Extract ALL payable records — this includes vendor invoices AND employee expense reports AND travel bookings AND utility statements.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DOCUMENT TYPES TO EXTRACT (all count as "invoice_type"):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. VENDOR INVOICES — traditional invoices with "Invoice #" or "Invoice Number"
   - invoice_number = the Invoice # value
   - total_amount   = "Invoice Total", "Total this Invoice", "Balance Due", "Total Due"
   - DO NOT use "Total Now Due" or "Total Outstanding" when those include prior balances

2. HOTEL GUEST FOLIOS — Hilton, Marriott, Doubletree, Hampton Inn, Element, Tru by Hilton etc.
   - invoice_number = "Confirmation Number" (e.g. 85148652) or "Folio ID" or "Guest Number"
   - total_amount   = the CREDIT CARD charge amount shown as "CREDIT CARD ($702.36)" or
                      "Total Charges" or room + tax subtotal. Use the absolute value.
   - vendor         = hotel name
   - If this is a continuation page (page 2 of 2) with charges but no guest name header,
     still extract it as page_type="continuation" with invoice_number and total_amount

3. EMPLOYEE EXPENSE REPORTS — pages showing "Report Name", "Report Id", "Employee Name",
   "Total Paid By Company", "Amount Due Employee"
   - invoice_number = the Report Id value (e.g. "6504FBA0E99A43258998")
   - total_amount   = "Total Paid By Company" value (preferred) OR "Amount Due Employee"
   - vendor         = "Employee Expense Report"
   - billed_to      = the Employee Name
   - invoice_date   = the Report Date
   - NOTE: Only extract the SUMMARY page (the one with "Total Paid By Company") —
     skip individual receipt pages (restaurant receipts, airline receipts, etc.)

4. UBER / LYFT BILLING STATEMENTS — pages with "Statement #", "Total", issued by Uber/Lyft
   - invoice_number = "Statement #" value (e.g. "53AD8A11F3")
   - total_amount   = the "Total" dollar amount on the statement summary page
   - vendor         = "Uber" or "Lyft"

5. TRAVEL BOOKING CONFIRMATIONS — American Express GBT, Concur, etc.
   Pages showing "Total Fare", "Invoice Booking Reference", "Trip ID"
   - invoice_number = "Invoice Booking Reference" or "Trip ID" value
   - total_amount   = "Total Fare : USD XXX.XX" value
   - vendor         = "American Express GBT" or the travel agency name
   - Only extract the page that shows the "Total Fare" — skip itinerary-only pages

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SKIP THESE (return page_type="other"):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Individual meal receipts (Wendy's, Starbucks, Chick-fil-A) — small POS receipts with no invoice #
- Airline boarding passes and baggage receipts
- Parking receipts under $50
- SAVI/OccuMedX billing report DETAIL pages (time-sheets with clock-in/clock-out rows)
  NOTE: the INVOICE page for SAVI/OccuMedX (the one with the invoice number and total) SHOULD be extracted
- Packing lists, order confirmations with no total
- Blank or near-blank pages

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GENERAL RULES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Dates: YYYY-MM-DD. If unparseable, keep original string.
- total_amount: plain number, no $ or commas.
- page_number: the === PAGE N === number for this record.
- page_type: "invoice" for new records | "continuation" for page 2+ with amounts | "other" to skip
- If a page has multiple separate payable records, return one entry per record.

Return ONLY a JSON array (no markdown, no code fences):
[
  {
    "invoice_number": "string or null",
    "invoice_date":   "YYYY-MM-DD or null",
    "vendor":         "string or null",
    "billed_to":      "string or null",
    "total_amount":   number or null,
    "page_number":    integer,
    "page_type":      "invoice" | "continuation" | "other"
  }
]
If nothing extractable, return [].
"""

def call_openai_with_retry(messages, max_retries=6):
    wait = 10
    for attempt in range(max_retries):
        try:
            resp = aoai_client.chat.completions.create(
                model=AOAI_DEPLOYMENT,
                messages=messages,
                temperature=0,
                max_tokens=30000
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if ("429" in err or "rate_limit" in err.lower()) and attempt < max_retries - 1:
                print(f"\n    [429] waiting {wait}s (attempt {attempt+1}/{max_retries})...", end="", flush=True)
                time.sleep(wait)
                wait = min(wait * 2, 120)
                continue
            raise

def extract_pages(page_dict):
    page_nums = sorted(page_dict.keys())
    sections  = [f"=== PAGE {p} ===\n{page_dict[p][:PAGE_TEXT_LIMIT]}" for p in page_nums]
    user_msg  = "\n\n".join(sections)

    raw = call_openai_with_retry([
        {"role": "system", "content": PAGE_EXTRACT_SYSTEM},
        {"role": "user",   "content": user_msg}
    ])
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    records = json.loads(raw)

    valid = []
    for r in records:
        ptype = r.get("page_type", "")
        if ptype == "invoice":
            pass
        elif ptype == "continuation" and r.get("total_amount"):
            pass  # keep continuation pages that carry an amount
        else:
            continue  # skip "other" and empty continuations

        inv_num = (r.get("invoice_number") or "").strip()
        if not inv_num:
            continue
        r["_source_page"]      = r.get("page_number") or page_nums[0]
        r["_is_continuation"]  = (ptype == "continuation")
        valid.append(r)
    return valid

def extract_batch(batch_num, page_batch):
    try:
        records = extract_pages(page_batch)
        return batch_num, records, []
    except Exception as e:
        print(f"\n  [FAIL] Batch {batch_num} (pg {min(page_batch)}-{max(page_batch)}): {str(e)[:80]}")
        return batch_num, [], list(page_batch.keys())

def extract_single_page(page_num, page_text):
    try:
        records = extract_pages({page_num: page_text})
        return page_num, records
    except Exception as e:
        print(f"\n  [FAIL] Single page {page_num}: {str(e)[:60]}")
        return page_num, []

# ── Build page batches ────────────────────────────────────────────────────────
sorted_pages  = sorted(page_texts.items())
total_pages   = len(sorted_pages)
page_batches  = []
for i in range(0, total_pages, PAGE_BATCH_SIZE):
    chunk = dict(sorted_pages[i : i + PAGE_BATCH_SIZE])
    page_batches.append((len(page_batches) + 1, chunk))
total_batches = len(page_batches)

print("=" * 65)
print(f"  Pages       : {total_pages}")
print(f"  Batch size  : {PAGE_BATCH_SIZE} pages/call  ->  {total_batches} batches total")
print(f"  Workers     : {EXTRACT_WORKERS} concurrent")
print(f"  Model       : {AOAI_DEPLOYMENT}  (captures invoices + expense reports + hotel folios + Uber)")
print("=" * 65)

# ── Checkpoint / resume ───────────────────────────────────────────────────────
CHECKPOINT_FILE = OUTPUT_CSV.replace(".csv", "_checkpoint.json")
all_page_records  = []
completed_batches = set()

if os.path.exists(CHECKPOINT_FILE):
    try:
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as cf:
            ckpt = json.load(cf)
        all_page_records  = ckpt.get("records", [])
        completed_batches = set(ckpt.get("done", []))
        print(f"\n  Checkpoint: {len(completed_batches)}/{total_batches} batches done, "
              f"{len(all_page_records)} records. Resuming...")
    except Exception as ce:
        print(f"  Checkpoint unreadable ({ce}), starting fresh.")

remaining    = [(n, b) for n, b in page_batches if n not in completed_batches]
failed_pages = []
print(f"  Batches to process: {len(remaining)}\n")

# ── Main extraction loop ──────────────────────────────────────────────────────
with concurrent.futures.ThreadPoolExecutor(max_workers=EXTRACT_WORKERS) as executor:
    futures = {executor.submit(extract_batch, n, b): n for n, b in remaining}
    for future in tqdm(concurrent.futures.as_completed(futures),
                       total=len(remaining), desc="Extracting batches"):
        batch_num, records, fail_pgs = future.result()
        all_page_records.extend(records)
        completed_batches.add(batch_num)
        failed_pages.extend(fail_pgs)
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as cf:
            json.dump({"records": all_page_records,
                       "done": list(completed_batches),
                       "failed_pages": failed_pages}, cf)
        time.sleep(BATCH_INTER_WAIT)

# ── Retry failed pages individually ──────────────────────────────────────────
if failed_pages:
    print(f"\n  Retrying {len(failed_pages)} individually failed pages...")
    time.sleep(20)
    for pnum in tqdm(sorted(set(failed_pages)), desc="Retry failed pages"):
        _, records = extract_single_page(pnum, page_texts[pnum])
        all_page_records.extend(records)
        time.sleep(4)
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as cf:
        json.dump({"records": all_page_records,
                   "done": list(completed_batches),
                   "failed_pages": []}, cf)

print(f"\nCheckpoint saved -> {os.path.basename(CHECKPOINT_FILE)}")

# ── Sort & Deduplicate ────────────────────────────────────────────────────────
all_page_records.sort(key=lambda r: (r["_source_page"], r.get("invoice_number") or ""))
print(f"Raw records (before dedup) : {len(all_page_records)}")

inv_groups = defaultdict(list)
for r in all_page_records:
    inv_num = (r.get("invoice_number") or "").strip()
    if inv_num:
        inv_groups[inv_num].append(r)

raw_invoices = []
for seq_idx, (inv_num, records) in enumerate(inv_groups.items(), start=1):
    all_pages = sorted({r["_source_page"] for r in records})
    pg_str    = ", ".join(str(p) for p in all_pages)

    invoice_records = [r for r in records if not r.get("_is_continuation")]

    # Best metadata: prefer non-continuation with date + vendor
    meta = sorted(invoice_records or records, key=lambda r: (
        r.get("invoice_date") is not None,
        r.get("vendor") is not None,
        r["_source_page"]
    ), reverse=True)[0]

    # Best amount: prefer non-continuation with amount, fall back to continuation
    best_amt = sorted(records, key=lambda r: (
        r.get("total_amount") is not None,
        not r.get("_is_continuation", False),
        r["_source_page"]
    ), reverse=True)[0]

    raw_invoices.append({
        "_idx"   : seq_idx,
        "invnum" : inv_num,
        "invdt"  : str(meta.get("invoice_date") or ""),
        "invto"  : str(meta.get("billed_to") or ""),
        "invamt" : f"{best_amt['total_amount']:.2f}" if best_amt.get("total_amount") else "",
        "pgnum"  : pg_str,
        "_vendor": str(meta.get("vendor") or ""),
        "_items" : "",
    })

print(f"Unique invoices/expenses (after dedup): {len(raw_invoices)}")
recovered = sum(1 for r in raw_invoices if r["invamt"])
print(f"Records with amount                   : {recovered} / {len(raw_invoices)}")

total_amt = sum(float(r["invamt"]) for r in raw_invoices if r["invamt"])
print(f"Total extracted amount                : USD {total_amt:,.2f}")

print(f"\nFirst 30 records:")
for inv in raw_invoices[:30]:
    print(f"  pg[{inv['pgnum']:>8}]  #{inv['invnum']:<25} {inv['invdt']}  ${inv['invamt']:>10}  {inv['_vendor']}")
if len(raw_invoices) > 30:
    print(f"  ... and {len(raw_invoices) - 30} more")


  Pages       : 808
  Batch size  : 5 pages/call  ->  162 batches total
  Workers     : 10 concurrent
  Model       : gpt-4.1-mini  (captures invoices + expense reports + hotel folios + Uber)
  Batches to process: 162



Extracting batches:  60%|██████    | 98/162 [00:47<00:15,  4.03it/s]


    [429] waiting 10s (attempt 1/6)...
    [429] waiting 10s (attempt 1/6)...

Extracting batches:  62%|██████▏   | 101/162 [00:48<00:25,  2.38it/s]


    [429] waiting 10s (attempt 1/6)...
    [429] waiting 10s (attempt 1/6)...

...


    [429] waiting 10s (attempt 1/6)...
    [429] waiting 10s (attempt 1/6)...

Extracting batches:  93%|█████████▎| 150/162 [01:30<00:10,  1.13it/s]


    [429] waiting 10s (attempt 1/6)...

Extracting batches: 100%|██████████| 162/162 [02:08<00:00,  1.26it/s]


Checkpoint saved -> invoices_output_full_checkpoint.json
Raw records (before dedup) : 509
Unique invoices/expenses (after dedup): 439
Records with amount                   : 436 / 439
Total extracted amount                : USD 1,205,939.63

First 30 records:
  pg[       1]  #INVE-119967               2026-01-15  $    369.21  ezCater, Inc
  pg[    3, 4]  #40279049                  2026-01-22  $    380.00  Duncan Parnell
  pg[       5]  #40279845                  2026-01-23  $     63.50  Duncan Parnell
  pg[       6]  #40275619                  2026-01-13  $    118.05  Duncan Parnell
  pg[       7]  #40280339                  2026-01-26  $     41.75  Duncan Parnell
  pg[       8]  #40280898                  2026-01-27  $    113.18  Duncan Parnell
  pg[       9]  #40283663                  2026-02-04  $     25.40  Duncan Parnell
  pg[      10]  #56396                     2026-01-26  $    470.00  Aerial Innovations Southeast
  pg[      11]  #56397                     2026-01-26  $    470

In [21]:
# ── Cell 5 · Step 3 — Enrich with category + description ─────────────────────
import time
from tqdm import tqdm

ENRICH_SYSTEM = """You are an expert accountant and expense analyst for a construction company.
Given a batch of expense/invoice summaries in JSON, return a JSON array with one object per record.
Each object must have exactly two keys:
  "invcat": a short category label. Choose from (or add your own if none fit):
    "Catering Services", "Equipment Rental", "Construction Supplies",
    "Office Supplies", "Professional Services", "Fuel & Gasoline",
    "Waste Management", "Travel & Accommodation", "Uniform & Facility Services",
    "Shipping & Delivery", "Occupational Health Services",
    "Environmental Health & Safety Services", "Hardware & Equipment",
    "Software Subscription", "Printing & Signage", "Utilities", "Marketing",
    "Parking Services", "Food & Beverage", "Aerial / Drone Services",
    "Employee Expense Reimbursement", "Transportation & Rideshare"
  "invdec": plain-English description, max 2 sentences.
Return only the JSON array, no extra text."""

def enrich_batch_safe(batch):
    wait = 12
    for attempt in range(7):
        try:
            prompt = json.dumps([{
                "id": inv["_idx"], "vendor": inv["_vendor"],
                "billed_to": inv["invto"], "date": inv["invdt"],
                "amount": inv["invamt"], "inv_num": inv["invnum"],
            } for inv in batch], indent=2)

            resp = aoai_client.chat.completions.create(
                model=AOAI_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": ENRICH_SYSTEM},
                    {"role": "user",   "content": prompt}
                ],
                temperature=0.2,
                max_tokens=1500
            )
            raw = resp.choices[0].message.content.strip()
            if raw.startswith("```"):
                raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
            enriched = json.loads(raw)
            return [{
                "_idx"   : batch[i]["_idx"],
                "invcat" : enriched[i].get("invcat", "Uncategorised") if i < len(enriched) else "Uncategorised",
                "invdec" : enriched[i].get("invdec", "") if i < len(enriched) else "",
            } for i in range(len(batch))]
        except Exception as e:
            err = str(e)
            if ("429" in err or "rate_limit" in err.lower()) and attempt < 6:
                print(f"\n  [429] enrichment, waiting {wait}s (attempt {attempt+1}/7)...", end="", flush=True)
                time.sleep(wait)
                wait = min(wait * 2, 120)
                continue
            print(f"\n  [WARN] enrich failed: {err[:60]}")
            return [{"_idx": inv["_idx"], "invcat": "Uncategorised", "invdec": ""} for inv in batch]
    return [{"_idx": inv["_idx"], "invcat": "Uncategorised", "invdec": ""} for inv in batch]

batches = [raw_invoices[i:i+BATCH_SIZE] for i in range(0, len(raw_invoices), BATCH_SIZE)]
print(f"Enriching {len(raw_invoices)} records across {len(batches)} batches (sequential + 2s gap)...")
print("Waiting 10s for rate limit to clear after extraction...")
time.sleep(10)

all_enriched = []
for batch in tqdm(batches, desc="Enriching"):
    all_enriched.extend(enrich_batch_safe(batch))
    time.sleep(2)

enrichment_map = {r["_idx"]: r for r in all_enriched}
uncategorised  = sum(1 for r in all_enriched if r["invcat"] == "Uncategorised")
print(f"\nEnrichment complete: {len(enrichment_map)} records | {uncategorised} uncategorised")

from collections import Counter
print("\nCategory breakdown:")
for cat, cnt in sorted(Counter(r["invcat"] for r in all_enriched).items(), key=lambda x: -x[1]):
    print(f"  {cnt:>4}  {cat}")


Enriching 439 records across 22 batches (sequential + 2s gap)...
Waiting 10s for rate limit to clear after extraction...


Enriching: 100%|██████████| 22/22 [04:50<00:00, 13.19s/it]


Enrichment complete: 439 records | 0 uncategorised

Category breakdown:
    94  Equipment Rental
    77  Transportation & Rideshare
    46  Construction Supplies
    36  Employee Expense Reimbursement
    32  Travel & Accommodation
    31  Waste Management
    26  Uniform & Facility Services
    23  Fuel & Gasoline
    17  Professional Services
    15  Office Supplies
    13  Food & Beverage
     8  Shipping & Delivery
     6  Environmental Health & Safety Services
     4  Printing & Signage
     3  Aerial / Drone Services
     3  Occupational Health Services
     3  Hardware & Equipment
     1  Catering Services
     1  Utilities


In [22]:
# ── Cell 6 · Step 4 — Assemble final DataFrame and save to CSV ────────────────
import pandas as pd
from datetime import datetime

def clean(val):
    if not val:
        return ""
    return " | ".join(p.strip() for p in str(val).splitlines() if p.strip())

rows = []
for i, inv in enumerate(raw_invoices, start=1):
    enrich = enrichment_map.get(inv["_idx"], {})
    rows.append({
        "Sr No"              : i,
        "Invoice Date"       : inv["invdt"],
        "Invoice By"         : clean(inv["_vendor"]),
        "Invoice To"         : clean(inv["invto"]),
        "Invoice Number"     : inv["invnum"],
        "Total Amount (USD)" : inv["invamt"],
        "Page Number"        : inv["pgnum"],
        "Category"           : enrich.get("invcat", ""),
        "Description"        : enrich.get("invdec", ""),
    })

COLUMNS = [
    "Sr No", "Invoice Date", "Invoice By", "Invoice To",
    "Invoice Number", "Total Amount (USD)", "Page Number",
    "Category", "Description"
]

df = pd.DataFrame(rows, columns=COLUMNS)

import os
save_path = OUTPUT_CSV
if os.path.exists(save_path):
    try:
        open(save_path, "a").close()
    except PermissionError:
        ts = datetime.now().strftime("%H%M%S")
        save_path = OUTPUT_CSV.replace(".csv", f"_{ts}.csv")
        print(f"  File open in Excel, saving as: {os.path.basename(save_path)}")

df.to_csv(save_path, index=False, encoding="utf-8-sig")

numeric_amt = pd.to_numeric(df["Total Amount (USD)"], errors="coerce")
grand_total = numeric_amt.sum()
missing_amt = numeric_amt.isna().sum()

print("=" * 60)
print(f"  CSV saved : {save_path}")
print(f"  Total rows: {len(df)}")
print(f"  With amount : {len(df) - missing_amt}   |   Missing amount: {missing_amt}")
print(f"  GRAND TOTAL : USD {grand_total:,.2f}")
print("=" * 60)
print("\nPreview (first 10 rows):")
df.head(10)


  File open in Excel, saving as: invoices_output_full_155245.csv
  CSV saved : C:\Users\KR614XU\Downloads\Ishaan\invoices_output_full_155245.csv
  Total rows: 439
  With amount : 436   |   Missing amount: 3
  GRAND TOTAL : USD 1,205,939.63

Preview (first 10 rows):


,Sr No,Invoice Date,Invoice By,Invoice To,Invoice Number,Total Amount (USD),Page Number,Category,Description
0,1,2026-01-15,"ezCater, Inc","HITT Contracting, Inc.",INVE-119967,369.21,1,Catering Services,Expense for catering services provided by ezCa...
1,2,2026-01-22,Duncan Parnell,Hitt Contracting Inc,40279049,380.00,"3, 4",Construction Supplies,Purchase of construction-related supplies from...
2,3,2026-01-23,Duncan Parnell,Hitt Contracting Inc,40279845,63.50,5,Construction Supplies,Additional purchase of construction materials ...
3,4,2026-01-13,Duncan Parnell,Hitt Contracting Inc,40275619,118.05,6,Construction Supplies,Further acquisition of construction supplies f...
4,5,2026-01-26,Duncan Parnell,Hitt Contracting Inc,40280339,41.75,7,Construction Supplies,Small purchase of construction materials from ...
5,6,2026-01-27,Duncan Parnell,Hitt Contracting Inc,40280898,113.18,8,Construction Supplies,Purchase of construction-related items from Du...
6,7,2026-02-04,Duncan Parnell,Hitt Contracting Inc,40283663,25.40,9,Construction Supplies,Minor purchase of construction supplies from D...
7,8,2026-01-26,Aerial Innovations Southeast,Robert Sorenson,56396,470.00,10,Aerial / Drone Services,Aerial drone services provided by Aerial Innov...
8,9,2026-01-26,Aerial Innovations Southeast,Robert Sorenson,56397,470.00,11,Aerial / Drone Services,Additional aerial drone service by Aerial Inno...
9,10,2026-02-03,Aerial Innovations Southeast,Robert Sorenson,56451,470.00,12,Aerial / Drone Services,Continued aerial drone services from Aerial In...
